# CEHARPS 05 — SHAP ช่วงความเชื่อมั่น และการคัดเลือกพื้นที่ / SHAP, uncertainty, and site selection

SHAP อธิบายความสัมพันธ์ภายในโมเดล ไม่ใช่หลักฐานเชิงเหตุและผล ส่วนต้นทุนใช้เป็นข้อจำกัดงบประมาณและไม่นับซ้ำในคะแนนประโยชน์


In [ ]:
import json  # TH: นำเข้าเครื่องมือ JSON | EN: Import JSON utilities.
import subprocess  # TH: นำเข้าเครื่องมือเรียกคำสั่งระบบ | EN: Import subprocess utilities.
import sys  # TH: นำเข้าข้อมูลตัวแปลภาษา Python | EN: Import Python runtime information.
from pathlib import Path  # TH: นำเข้าคลาสจัดการพาธ | EN: Import the path-management class.
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "shap>=0.50,<1", "pulp>=3,<4", "xgboost>=3.1,<4", "scikit-learn>=1.5,<2", "joblib>=1.4,<2"])  # TH: ติดตั้งไลบรารีอธิบายและ optimization | EN: Install explanation and optimization libraries.
import joblib  # TH: นำเข้าเครื่องมือโหลดโมเดล | EN: Import model-loading utilities.
import matplotlib.pyplot as plt  # TH: นำเข้าเครื่องมือสร้างกราฟ | EN: Import plotting utilities.
import numpy as np  # TH: นำเข้า NumPy | EN: Import NumPy.
import pandas as pd  # TH: นำเข้า pandas | EN: Import pandas.
import pulp  # TH: นำเข้าเครื่องมือแก้ปัญหางบประมาณ | EN: Import budget-optimization tools.
import shap  # TH: นำเข้า SHAP | EN: Import SHAP.
from google.colab import drive  # TH: นำเข้าเครื่องมือเชื่อม Drive | EN: Import the Drive connector.
from sklearn.base import clone  # TH: นำเข้าเครื่องมือคัดลอกโมเดลแบบยังไม่ฝึก | EN: Import unfitted model cloning.
drive.mount("/content/drive")  # TH: เชื่อม Google Drive | EN: Mount Google Drive.
PROJECT_ROOT = Path("/content/drive/MyDrive/CEHARPS")  # TH: กำหนดโฟลเดอร์โครงการ | EN: Define the project folder.
CONFIG = json.loads((PROJECT_ROOT / "config.json").read_text(encoding="utf-8"))  # TH: อ่านค่ากลาง | EN: Load shared settings.
META = json.loads((PROJECT_ROOT / "data/processed/health_metadata.json").read_text(encoding="utf-8"))  # TH: อ่านเมทาดาทาข้อมูล | EN: Load data metadata.
MODEL_META = json.loads((PROJECT_ROOT / "artifacts/health/selected_health_model.json").read_text(encoding="utf-8"))  # TH: อ่านเมทาดาทาโมเดล | EN: Load model metadata.
PIPELINE = joblib.load(PROJECT_ROOT / "artifacts/health/selected_health_model.joblib")  # TH: โหลดโมเดลสุขภาพที่เลือก | EN: Load the selected health model.
FEATURES = list(META["features"])  # TH: อ่านรายชื่อตัวแปร | EN: Read the feature list.
TARGET = str(META["target"])  # TH: อ่านชื่อเป้าหมาย | EN: Read the target name.
SEED = int(CONFIG["seed"])  # TH: อ่านค่าเมล็ดสุ่ม | EN: Read the random seed.


In [ ]:
processed = PROJECT_ROOT / "data/processed"  # TH: กำหนดโฟลเดอร์ข้อมูลพร้อมฝึก | EN: Define the processed-data folder.
train = pd.read_csv(processed / "health_train.csv")  # TH: อ่านชุดฝึก | EN: Load the training split.
test = pd.read_csv(processed / "health_test.csv")  # TH: อ่านชุดทดสอบ | EN: Load the test split.
imputer = PIPELINE.named_steps["imputer"]  # TH: ดึงตัวเติมค่าหายจาก pipeline | EN: Retrieve the fitted imputer.
model = PIPELINE.named_steps["model"]  # TH: ดึงโมเดลต้นไม้จาก pipeline | EN: Retrieve the fitted tree model.
background_raw = train[FEATURES].sample(n=min(200, len(train)), random_state=SEED)  # TH: สุ่มข้อมูลพื้นหลังจากชุดฝึกเท่านั้น | EN: Sample SHAP background from training data only.
sample_raw = test[FEATURES].sample(n=min(200, len(test)), random_state=SEED)  # TH: สุ่มข้อมูลที่ต้องการอธิบายจาก test | EN: Sample test rows to explain.
background = pd.DataFrame(imputer.transform(background_raw), columns=FEATURES)  # TH: เติมค่าหายของข้อมูลพื้นหลัง | EN: Impute the background data.
sample = pd.DataFrame(imputer.transform(sample_raw), columns=FEATURES)  # TH: เติมค่าหายของข้อมูลอธิบาย | EN: Impute the explained data.
explainer = shap.TreeExplainer(model, data=background, feature_perturbation="interventional")  # TH: สร้าง Tree SHAP พร้อมข้อมูลพื้นหลัง | EN: Create an interventional Tree SHAP explainer.
explanation = explainer(sample)  # TH: คำนวณค่า SHAP ของตัวอย่าง | EN: Calculate SHAP values for the sample.
shap.plots.beeswarm(explanation, max_display=min(15, len(FEATURES)), show=False)  # TH: สร้างกราฟสรุป SHAP | EN: Create a SHAP beeswarm summary.
plt.title(f"SHAP summary — {TARGET} ({MODEL_META['selected_model']})")  # TH: ตั้งชื่อกราฟ | EN: Set the plot title.
plt.tight_layout()  # TH: จัดระยะองค์ประกอบกราฟ | EN: Tighten the plot layout.
report_dir = PROJECT_ROOT / "reports"  # TH: กำหนดโฟลเดอร์รายงาน | EN: Define the report folder.
report_dir.mkdir(parents=True, exist_ok=True)  # TH: สร้างโฟลเดอร์รายงาน | EN: Create the report folder.
plt.savefig(report_dir / "shap_health_summary.png", dpi=200, bbox_inches="tight")  # TH: บันทึกกราฟ SHAP | EN: Save the SHAP plot.
plt.show()  # TH: แสดงกราฟใน Colab | EN: Display the plot in Colab.


In [ ]:
rng = np.random.default_rng(SEED)  # TH: สร้างตัวสุ่มสำหรับ bootstrap | EN: Create a bootstrap random generator.
unique_blocks = train["spatial_block"].astype(str).unique()  # TH: อ่านบล็อกเชิงพื้นที่ของชุดฝึก | EN: Read unique training spatial blocks.
predictions = []  # TH: เตรียมรายการผลพยากรณ์แต่ละโมเดล | EN: Initialize bootstrap predictions.
for member in range(int(CONFIG["bootstrap_models"])):  # TH: วนฝึกโมเดล bootstrap | EN: Iterate through bootstrap members.
    sampled_blocks = rng.choice(unique_blocks, size=len(unique_blocks), replace=True)  # TH: สุ่มบล็อกพร้อมคืนค่า | EN: Sample spatial blocks with replacement.
    bootstrap = pd.concat([train.loc[train["spatial_block"].astype(str) == block] for block in sampled_blocks], ignore_index=True)  # TH: รวมแถวตามบล็อกที่สุ่มรวมทั้งบล็อกซ้ำ | EN: Concatenate sampled blocks including repeats.
    member_model = clone(PIPELINE)  # TH: สร้าง pipeline ใหม่ที่ยังไม่ฝึก | EN: Clone an unfitted pipeline.
    member_model.fit(bootstrap[FEATURES], bootstrap[TARGET])  # TH: ฝึกด้วยตัวอย่าง bootstrap | EN: Fit on the block-bootstrap sample.
    predictions.append(np.clip(member_model.predict(test[FEATURES]), 0, 100))  # TH: เก็บผลพยากรณ์ test ช่วง 0–100 | EN: Store clipped test predictions.
matrix = np.vstack(predictions)  # TH: รวมผลโมเดลเป็นเมทริกซ์ | EN: Stack member predictions into a matrix.
uncertainty = test[[column for column in ["sample_id", "site_id", "spatial_block", "data_status"] if column in test.columns]].copy()  # TH: สร้างตารางผลพร้อมรหัส | EN: Build a traceable uncertainty table.
uncertainty[f"{TARGET}_actual"] = test[TARGET].to_numpy()  # TH: เพิ่มค่าจริง | EN: Add observed values.
uncertainty[f"{TARGET}_mean"] = matrix.mean(axis=0)  # TH: เพิ่มค่าพยากรณ์เฉลี่ย | EN: Add mean predictions.
uncertainty[f"{TARGET}_lower_95"] = np.quantile(matrix, 0.025, axis=0)  # TH: เพิ่มขอบล่าง 95% | EN: Add the lower 95% bound.
uncertainty[f"{TARGET}_upper_95"] = np.quantile(matrix, 0.975, axis=0)  # TH: เพิ่มขอบบน 95% | EN: Add the upper 95% bound.
uncertainty["uncertainty_width"] = uncertainty[f"{TARGET}_upper_95"] - uncertainty[f"{TARGET}_lower_95"]  # TH: คำนวณความกว้างช่วง | EN: Calculate interval width.
uncertainty.to_csv(PROJECT_ROOT / "artifacts/health/health_uncertainty.csv", index=False)  # TH: บันทึกผลความไม่แน่นอน | EN: Save uncertainty results.
print(uncertainty.head())  # TH: แสดงตัวอย่างผล | EN: Display example results.


In [ ]:
candidate_path = PROJECT_ROOT / "data/health/restoration_candidates.csv"  # TH: กำหนดไฟล์พื้นที่ฟื้นฟูจริง | EN: Define the real restoration-candidate file.
if candidate_path.exists():  # TH: ตรวจว่ามีข้อมูลพื้นที่จริงหรือไม่ | EN: Check whether real candidate data exists.
    candidates = pd.read_csv(candidate_path)  # TH: อ่านข้อมูลพื้นที่จริง | EN: Load real candidate data.
else:  # TH: สร้างตัวอย่างเมื่อยังไม่มีข้อมูลจริง | EN: Build a demonstration table when real data is absent.
    candidates = uncertainty[["site_id", f"{TARGET}_mean", "uncertainty_width"]].rename(columns={f"{TARGET}_mean": "health_score"}).copy()  # TH: เริ่มตารางตัวอย่างจากผลสุขภาพ | EN: Start demonstration candidates from health outputs.
    candidates["pressure_score"] = np.linspace(30, 80, len(candidates))  # TH: สร้างแรงกดดันจำลอง | EN: Create demonstration pressure scores.
    candidates["ecological_value"] = np.linspace(80, 40, len(candidates))  # TH: สร้างคุณค่าเชิงนิเวศจำลอง | EN: Create demonstration ecological values.
    candidates["feasibility"] = 65.0  # TH: กำหนดความเป็นไปได้จำลอง | EN: Set demonstration feasibility.
    candidates["cost"] = np.linspace(80000, 220000, len(candidates))  # TH: สร้างต้นทุนจำลอง | EN: Create demonstration costs.
    candidates["data_status"] = "synthetic_demo_not_for_decision"  # TH: ติดป้ายห้ามใช้ตัดสินใจจริง | EN: Label the table as non-operational demo data.
required = {"site_id", "health_score", "pressure_score", "ecological_value", "feasibility", "uncertainty_width", "cost"}  # TH: กำหนดคอลัมน์พื้นที่ที่จำเป็น | EN: Define required candidate columns.
missing = required.difference(candidates.columns)  # TH: หาคอลัมน์พื้นที่ที่ขาด | EN: Find missing candidate columns.
if missing:  # TH: ตรวจว่าขาดคอลัมน์หรือไม่ | EN: Check for missing columns.
    raise ValueError(f"Missing candidate columns: {sorted(missing)}")  # TH: แจ้งและหยุดเมื่อข้อมูลไม่ครบ | EN: Report and stop on incomplete data.
candidates["priority_benefit"] = 0.35 * (100 - candidates["health_score"]) + 0.25 * candidates["pressure_score"] + 0.25 * candidates["ecological_value"] + 0.15 * candidates["feasibility"] - 0.10 * candidates["uncertainty_width"]  # TH: คำนวณประโยชน์โดยไม่ใส่ต้นทุนซ้ำในคะแนน | EN: Calculate benefit without double-counting cost.
problem = pulp.LpProblem("CEHARPS_restoration", pulp.LpMaximize)  # TH: สร้างปัญหา optimization แบบเพิ่มประโยชน์สูงสุด | EN: Create a benefit-maximization problem.
choices = [pulp.LpVariable(f"select_{index}", cat="Binary") for index in candidates.index]  # TH: สร้างตัวแปรเลือกพื้นที่ 0 หรือ 1 | EN: Create binary site-selection variables.
problem += pulp.lpSum(candidates.loc[index, "priority_benefit"] * choices[position] for position, index in enumerate(candidates.index))  # TH: กำหนดเป้าหมายผลประโยชน์รวมสูงสุด | EN: Maximize total restoration benefit.
problem += pulp.lpSum(candidates.loc[index, "cost"] * choices[position] for position, index in enumerate(candidates.index)) <= float(CONFIG["restoration_budget"])  # TH: จำกัดต้นทุนรวมไม่เกินงบ | EN: Constrain total cost to the budget.
status = problem.solve(pulp.PULP_CBC_CMD(msg=False))  # TH: แก้ปัญหาด้วย CBC | EN: Solve the problem with CBC.
if pulp.LpStatus[status] != "Optimal":  # TH: ตรวจว่าสามารถหาคำตอบเหมาะที่สุดได้หรือไม่ | EN: Check for an optimal solution.
    raise RuntimeError(f"Optimization status: {pulp.LpStatus[status]}")  # TH: หยุดเมื่อคำตอบไม่สมบูรณ์ | EN: Stop on a non-optimal solution.
candidates["selected"] = [int(variable.value()) for variable in choices]  # TH: บันทึกผลเลือกพื้นที่ | EN: Record selected sites.
candidates = candidates.sort_values(["selected", "priority_benefit"], ascending=[False, False])  # TH: เรียงพื้นที่ที่เลือกและคะแนน | EN: Sort selected sites and benefit scores.
candidates.to_csv(PROJECT_ROOT / "artifacts/restoration_selection.csv", index=False)  # TH: บันทึกผลการคัดเลือกพื้นที่ | EN: Save the site-selection result.
print(candidates.loc[candidates["selected"] == 1, ["site_id", "priority_benefit", "cost"]])  # TH: แสดงชุดพื้นที่ที่เลือก | EN: Display the selected site portfolio.
